In [41]:
import requests
from dotenv import load_dotenv
import os
import pandas as pd
import time

In [42]:


load_dotenv()

DATABRICKS_INSTANCE = os.getenv("DATABRICKS_INSTANCE")
USER_AUTHENTICATION_TOKEN = os.getenv("USER_AUTHENTICATION_TOKEN")

url = f"https://{DATABRICKS_INSTANCE}/api/2.0/genie/spaces"
headers = {
    "Authorization": f"Bearer {USER_AUTHENTICATION_TOKEN}"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

response.json()

{'spaces': [{'space_id': '01f12a06040d16e382c5a0acda573523',
   'title': 'Agentic Email - Audience Agent',
   'description': 'This space provides insights into user behavior on Xome.com by tracking user interactions, property views, and auction activities across various pages.\n\nThis enables business users to retrieve and explore audience listings using simple natural language requests. It translates plain-English instructions such as requesting customers that recently viewed, saved, or bid on properties, and generate queries against the user behavior database.',
   'warehouse_id': '145ee66c3a851a47'},
  {'space_id': '01f11ea6ec63196c8044b11125caf55a',
   'title': 'Ginnie Mae Data Analysis',
   'warehouse_id': '145ee66c3a851a47'},
  {'space_id': '01f11dd4c2dc125b864fea7965a09927',
   'title': 'Ginnie Mae Loan Performance Analytics',
   'warehouse_id': '145ee66c3a851a47'},
  {'space_id': '01f0fb39c5e711b1816651c49fa0f0ee',
   'title': 'Audinece - Property Agent',
   'description': 'The

Some example questions:

Retrieve customer that saved at least 2 properties in the past 7 days. Find the last 2 properties they saved and last 2 properties they viewed.

Retrieve the last 4 saved or viewed properties in the past 7 days for each customer. Deduplicate between saved and viewed list giving saved higher priority. If there are overlaps, keep properties in saved and continue searching for viewed to fill the list. Label each property with their action type.

Retrieve customers that registered for any auction events in the past 7 days. Get top 2 properties for those events.

Get 4 least engaged (save or view) properties for the upcoming auction events. Write highlights for each property encouraging customer engagements, do not exceed 50 words.

In [43]:
question = """
    Retrieve customer that saved at least 2 properties in the past 7 days. 
    Find the last 2 properties they saved and last 2 properties they viewed.
    Get property details from dbo_listing including ListingPrice, StreetNumber,
    StreetName, City, State, County, TransactionType, SQFT
"""

In [44]:
# Start a conversation

GENIE_SPACE_ID = os.getenv("GENIE_SPACE_ID")

url = f"https://{DATABRICKS_INSTANCE}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/start-conversation"
headers = {
    "Authorization": f"Bearer {USER_AUTHENTICATION_TOKEN}"
}
payload = {
    "content": question
}

response = requests.post(url, headers=headers, json=payload)
response.raise_for_status()

data = response.json()
conversation_id = data["conversation_id"]
message_id = data["message_id"]

In [45]:
# Get message details
url = f"https://{DATABRICKS_INSTANCE}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/conversations/{conversation_id}/messages/{message_id}"
headers = {
    "Authorization": f"Bearer {USER_AUTHENTICATION_TOKEN}"
}

while True:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    data = response.json()
    status = data.get("status")
    print(f"Status: {status}")
    if status == "COMPLETED":
        break
    time.sleep(5)

Status: FILTERING_CONTEXT
Status: ASKING_AI
Status: PENDING_WAREHOUSE
Status: PENDING_WAREHOUSE
Status: ASKING_AI
Status: COMPLETED


In [ ]:
query = data["attachments"][0]["query"]["query"]
attachment_id = data["attachments"][0]["attachment_id"]
statement_id = data["query_result"]["statement_id"]
summary = data["attachments"][3]["text"]["content"]

In [ ]:
base_url = f"https://{DATABRICKS_INSTANCE}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/conversations/{conversation_id}/messages/{message_id}/query-result/{attachment_id}"
headers = {
    "Authorization": f"Bearer {USER_AUTHENTICATION_TOKEN}"
}

response = requests.get(base_url, headers=headers)
response.raise_for_status()
first = response.json()

statement = first["statement_response"]
columns = [col["name"] for col in statement["manifest"]["schema"]["columns"]]
total_chunks = statement["manifest"].get("total_chunk_count", 1)

all_rows = list(statement["result"]["data_array"])
# df = pd.DataFrame(all_rows, columns = columns)
# df.head()

,xome_user_id,event_type,listing_id,event_datetime,ListingPrice,StreetNumber,StreetName,City,State,County,TransactionType,SQFT
0,1613767,saved,415995960,2026-03-27 22:45:44.333831 UTC,110000,252,BURGOYNE,Norfolk,VA,NORFOLK CITY,NEWLY_FORECLOSED,810
1,1613767,saved,414412319,2026-03-27 22:45:38.391251 UTC,125000,1831,LASALLE,Portsmouth,VA,PORTSMOUTH CITY,NEWLY_FORECLOSED,1646
2,1613767,viewed,414704605,2026-03-27 22:30:59.394634 UTC,125000,10211,ELOKOMIN,Richmond,VA,CHESTERFIELD,REO,2109
3,1613767,viewed,412057248,2026-03-27 22:29:42.224078 UTC,330000,15606,Talland,Chesterfield,VA,CHESTERFIELD,NEWLY_FORECLOSED,3932
4,1621230,saved,417665350,2026-03-27 05:41:48.577423 UTC,325680,13849,JACKSON,Hesperia,CA,SAN BERNARDINO,FORECLOSURE_TRUSTEE,2024


In [50]:
# Fetch additional chunks if any
for chunk_index in range(1, total_chunks):
    chunk_url = f"https://{DATABRICKS_INSTANCE}/api/2.0/sql/statements/{statement_id}/result/chunks/{chunk_index}"
    chunk_response = requests.get(chunk_url, headers=headers)
    chunk_response.raise_for_status()
    all_rows.extend(chunk_response.json()["data_array"])

# Save query result into a json object
result_json = [dict(zip(columns, row)) for row in all_rows]

# Save query result into a dataframe
df = pd.DataFrame(all_rows, columns=columns)
df.head()

,xome_user_id,event_type,listing_id,event_datetime,ListingPrice,StreetNumber,StreetName,City,State,County,TransactionType,SQFT
0,1613767,saved,415995960,2026-03-27 22:45:44.333831 UTC,110000,252,BURGOYNE,Norfolk,VA,NORFOLK CITY,NEWLY_FORECLOSED,810
1,1613767,saved,414412319,2026-03-27 22:45:38.391251 UTC,125000,1831,LASALLE,Portsmouth,VA,PORTSMOUTH CITY,NEWLY_FORECLOSED,1646
2,1613767,viewed,414704605,2026-03-27 22:30:59.394634 UTC,125000,10211,ELOKOMIN,Richmond,VA,CHESTERFIELD,REO,2109
3,1613767,viewed,412057248,2026-03-27 22:29:42.224078 UTC,330000,15606,Talland,Chesterfield,VA,CHESTERFIELD,NEWLY_FORECLOSED,3932
4,1621230,saved,417665350,2026-03-27 05:41:48.577423 UTC,325680,13849,JACKSON,Hesperia,CA,SAN BERNARDINO,FORECLOSURE_TRUSTEE,2024


In [74]:
def genie_audience_property_query(question, conversation_id=None):
    import requests
    from dotenv import load_dotenv
    import os
    import pandas as pd
    import time
    
    load_dotenv()

    DATABRICKS_INSTANCE = os.getenv("DATABRICKS_INSTANCE")
    USER_AUTHENTICATION_TOKEN = os.getenv("USER_AUTHENTICATION_TOKEN")
    GENIE_SPACE_ID = os.getenv("GENIE_SPACE_ID")

    headers = {"Authorization": f"Bearer {USER_AUTHENTICATION_TOKEN}"}
    payload = {"content": question}   

    # Start or continue a conversation
    if conversation_id:
        url = f"https://{DATABRICKS_INSTANCE}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/conversations/{conversation_id}/messages"
    else:
        url = f"https://{DATABRICKS_INSTANCE}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/start-conversation"


    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    conversation_id = data["conversation_id"]
    message_id = data["message_id"]

    # Poll until COMPLETED
    url = f"https://{DATABRICKS_INSTANCE}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/conversations/{conversation_id}/messages/{message_id}"
    while True:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()
        status = data.get("status")
        print(f"Status: {status}")
        if status == "COMPLETED":
            break
        time.sleep(5)

    # Extract IDs
    query = data["attachments"][0]["query"]["query"]
    attachment_id = data["attachments"][0]["attachment_id"]
    statement_id = data["query_result"]["statement_id"]
    # summary = data["attachments"][3]["text"]["content"]

    # Fetch first chunk of query result
    base_url = f"https://{DATABRICKS_INSTANCE}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/conversations/{conversation_id}/messages/{message_id}/query-result/{attachment_id}"
    response = requests.get(base_url, headers=headers)
    response.raise_for_status()
    first = response.json()

    statement = first["statement_response"]
    columns = [col["name"] for col in statement["manifest"]["schema"]["columns"]]
    total_chunks = statement["manifest"].get("total_chunk_count", 1)
    all_rows = list(statement["result"]["data_array"])

    # Fetch additional chunks if any
    for chunk_index in range(1, total_chunks):
        chunk_url = f"https://{DATABRICKS_INSTANCE}/api/2.0/sql/statements/{statement_id}/result/chunks/{chunk_index}"
        chunk_response = requests.get(chunk_url, headers=headers)
        chunk_response.raise_for_status()
        all_rows.extend(chunk_response.json()["data_array"])

    # Build outputs
    result_json = [dict(zip(columns, row)) for row in all_rows]
    df = pd.DataFrame(all_rows, columns=columns)

    return {"query": query, 
            "result_table": df, 
            "result_json": result_json, 
            "conversation_id": conversation_id
            }


In [72]:
question = """
    Retrieve customers that registered for any auction events in the past 7 days. 
    Get top 2 properties for those events.
"""

In [75]:
result = genie_audience_property_query(question)

Status: FILTERING_CONTEXT
Status: ASKING_AI
Status: ASKING_AI
Status: ASKING_AI
Status: COMPLETED


In [78]:
conversation_id = result["conversation_id"]

In [79]:
followup = "how many unique properties are there in the output table"
followup_result = genie_audience_property_query(followup,conversation_id)

Status: FILTERING_CONTEXT
Status: ASKING_AI
Status: PENDING_WAREHOUSE
Status: ASKING_AI
Status: ASKING_AI
Status: COMPLETED
